# Streaming Gold Layer

## Overview

The Streaming Gold layer is the business-oriented analytical layer of the streaming branch of the Ubuntu Retail Group unified retail data platform.

The overall platform follows a unified **Bronze → Silver → Gold Medallion Architecture** for both batch and streaming workloads.

The Gold stage has two different implementations:

- **Batch Gold:** implemented in the Fabric Warehouse using SQL/T-SQL tables.
- **Streaming Gold:** implemented in the Fabric Lakehouse using Delta tables and PySpark Structured Streaming.

This does not represent two separate Medallion Architectures. It is one logical Medallion Architecture with separate Gold implementations for the batch and streaming workloads.

The Eventhouse/KQL branch is a separate, parallel Real-Time Intelligence path and is not part of the Medallion Architecture.

---

# 1. Streaming Architecture

The streaming pipeline follows this architecture:

```text
Python Event Simulator
        |
        v
Azure Event Hubs
        |
        v
Microsoft Fabric Eventstream
        |
        v
Lakehouse Bronze
        |
        v
PySpark Structured Streaming
        |
        v
Streaming Silver
        |
        v
PySpark Structured Streaming
        |
        v
Streaming Gold
        |
        v
Power BI


### 1. Inspect Streaming Silver

Before building the Streaming Gold layer, inspect the existing Silver streaming table to confirm its schema and sample records.

The Silver table is the source for all Streaming Gold transformations. At this stage, no transformations are applied; the purpose is to verify that the expected columns and data are available.

In [3]:
from pyspark.sql import functions as F

# Inspect the existing Streaming Silver table
df = spark.read.table("2_silver.silver_streaming_events")

df.printSchema()
df.show(10, truncate=False)


StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 5, Finished, Available, Finished, False)

root
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- view_duration_seconds: long (nullable = true)
 |-- EventProcessedUtcTime: timestamp (nullable = true)
 |-- PartitionId: long (nullable = true)
 |-- EventEnqueuedUtcTime: timestamp (nullable = true)
 |-- event_key: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- processing_delay_seconds: long (nullable = true)

+--------------+--------------------------+-----------+----------+---------------------------------+--------------+----------+---------------------+---------------------+-----------+--------------------+----------------------------------------------------------------+----------+----------+--------------

### 2. Create Streaming DataFrame

Create a Spark Structured Streaming DataFrame from the existing Silver streaming table.

The Silver layer has already performed the required data cleaning, validation, event-time processing, watermarking, and deduplication. Therefore, the Streaming Gold layer will use the validated Silver stream as its source and focus on producing analytical metrics for downstream reporting.

The streaming DataFrame will continuously receive new records as they are added to the Silver streaming table.

In [4]:
# Create the Streaming DataFrame from the Silver layer
silver_stream = (
    spark.readStream
         .table("2_silver.silver_streaming_events")
)

silver_stream.printSchema()

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 6, Finished, Available, Finished, False)

root
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- view_duration_seconds: long (nullable = true)
 |-- EventProcessedUtcTime: timestamp (nullable = true)
 |-- PartitionId: long (nullable = true)
 |-- EventEnqueuedUtcTime: timestamp (nullable = true)
 |-- event_key: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- processing_delay_seconds: long (nullable = true)



### 3. Streaming Gold – Event Metrics

Create the first Streaming Gold analytical table containing event activity over time.

Incoming Silver events are grouped into one-minute event-time windows and aggregated by event type. This produces a continuously updated view of how many events of each type are occurring per minute.

The resulting Gold table will support real-time monitoring of event volumes and event-type trends in Power BI.

A 10-minute watermark is applied to control streaming state and allow limited late-arriving events to be processed.

In [5]:
# Create one-minute event metrics by event type
event_metrics_stream = (
    silver_stream
    .withWatermark("event_timestamp", "10 minutes")
    .groupBy(
        F.window("event_timestamp", "1 minute"),
        F.col("event_type")
    )
    .count()
    .withColumnRenamed("count", "event_count")
    .select(
        F.col("window.start").alias("event_minute"),
        F.col("event_type"),
        F.col("event_count")
    )
)

event_metrics_stream.printSchema()

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 7, Finished, Available, Finished, False)

root
 |-- event_minute: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_count: long (nullable = false)



### 4. Streaming Gold – Country Metrics

Create a Streaming Gold analytical table containing event activity by country.

Incoming Silver events are aggregated by country to provide a high-level view of geographic activity in the streaming data.

This metric will support geographic monitoring and visualization in Power BI.

The country aggregation uses the validated Silver streaming data and does not repeat the data-quality and deduplication logic already performed in the Silver layer.

In [6]:
# Create event metrics by country
country_metrics_stream = (
    silver_stream
    .withWatermark("event_timestamp", "10 minutes")
    .groupBy(
        F.col("country")
    )
    .count()
    .withColumnRenamed("count", "event_count")
    .select(
        F.col("country"),
        F.col("event_count")
    )
)

country_metrics_stream.printSchema()

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 8, Finished, Available, Finished, False)

root
 |-- country: string (nullable = true)
 |-- event_count: long (nullable = false)



### 5. Streaming Gold – Real-Time KPIs

Create a consolidated Streaming Gold table containing key real-time streaming metrics.

Events are aggregated into one-minute event-time windows to produce total event volume and event-type metrics for each minute.

The KPI table includes:

- Total events
- Product views
- Cart actions
- Checkout starts
- Successful purchases
- Checkout-to-purchase conversion rate
- Average processing delay

The checkout-to-purchase conversion rate is calculated as:

Checkout-to-Purchase Rate = Purchases / Checkout Starts × 100

Revenue-based metrics are intentionally excluded because the unified Silver streaming table does not contain purchase revenue attributes such as quantity, unit price, or total order value.

In [7]:
# Create one-minute real-time KPI metrics
realtime_kpis_stream = (
    silver_stream
    .withWatermark("event_timestamp", "10 minutes")
    .groupBy(
        F.window("event_timestamp", "1 minute")
    )
    .agg(
        F.count("*").alias("total_events"),
        F.sum(
            F.when(F.col("event_type") == "product_view", 1).otherwise(0)
        ).alias("product_views"),
        F.sum(
            F.when(F.col("event_type") == "cart_action", 1).otherwise(0)
        ).alias("cart_actions"),
        F.sum(
            F.when(F.col("event_type") == "checkout_start", 1).otherwise(0)
        ).alias("checkout_starts"),
        F.sum(
            F.when(F.col("event_type") == "purchase_success", 1).otherwise(0)
        ).alias("purchases"),
        F.avg("processing_delay_seconds").alias(
            "avg_processing_delay_seconds"
        )
    )
    .withColumn(
        "checkout_to_purchase_rate",
        F.when(
            F.col("checkout_starts") > 0,
            (F.col("purchases") / F.col("checkout_starts")) * 100
        ).otherwise(0.0)
    )
    .select(
        F.col("window.start").alias("event_minute"),
        F.col("total_events"),
        F.col("product_views"),
        F.col("cart_actions"),
        F.col("checkout_starts"),
        F.col("purchases"),
        F.col("checkout_to_purchase_rate"),
        F.col("avg_processing_delay_seconds")
    )
)

realtime_kpis_stream.printSchema()

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 9, Finished, Available, Finished, False)

root
 |-- event_minute: timestamp (nullable = true)
 |-- total_events: long (nullable = false)
 |-- product_views: long (nullable = true)
 |-- cart_actions: long (nullable = true)
 |-- checkout_starts: long (nullable = true)
 |-- purchases: long (nullable = true)
 |-- checkout_to_purchase_rate: double (nullable = true)
 |-- avg_processing_delay_seconds: double (nullable = true)



### 6. Configure Streaming Gold Sinks

Configure the Structured Streaming outputs that write the transformed Gold metrics to Delta tables in the `3_gold` Lakehouse schema.

Each Gold table uses a dedicated checkpoint location so that Spark can track streaming progress and maintain state independently for each aggregation.

The Gold tables created by this section are:

- `3_gold.gold_streaming_event_metrics`
- `3_gold.gold_streaming_country_metrics`
- `3_gold.gold_streaming_realtime_kpis`

The streams use update mode because the aggregated results for an event-time window can be updated as additional events arrive within the watermark period.

In [8]:
from delta.tables import DeltaTable

def write_event_metrics(batch_df, batch_id):
    target_table = "3_gold.gold_streaming_event_metrics"

    if not spark.catalog.tableExists(target_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("target")
            .merge(
                batch_df.alias("source"),
                """
                target.event_minute = source.event_minute
                AND target.event_type = source.event_type
                """
            )
            .whenMatchedUpdate(set={
                "event_count": "source.event_count"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 10, Finished, Available, Finished, False)

In [9]:
event_metrics_query = (
    event_metrics_stream.writeStream
    .outputMode("update")
    .foreachBatch(write_event_metrics)
    .option(
        "checkpointLocation",
        "Files/checkpoints/gold_event_metrics"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 11, Finished, Available, Finished, False)

In [10]:
# Check active Streaming Gold queries
for query in spark.streams.active:
    print(query.name, query.status)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 12, Finished, Available, Finished, False)

None {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [11]:
# Check all active streaming queries
for query in spark.streams.active:
    print("Name:", query.name)
    print("Status:", query.status)
    print()

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 13, Finished, Available, Finished, False)

Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}



In [12]:
spark.read.table("3_gold.gold_streaming_event_metrics").show(
    20,
    truncate=False
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 14, Finished, Available, Finished, False)

+-------------------+----------------+-----------+
|event_minute       |event_type      |event_count|
+-------------------+----------------+-----------+
|2026-08-12 17:07:00|purchase_success|5          |
|2026-08-12 15:09:00|cart_action     |6          |
|2026-09-03 11:23:00|checkout_start  |2          |
|2026-09-03 11:21:00|product_view    |9          |
|2026-09-03 11:40:00|checkout_start  |7          |
|2026-09-03 11:35:00|purchase_success|4          |
|2026-08-12 16:11:00|purchase_success|4          |
|2026-08-12 15:33:00|inventory_update|3          |
|2026-08-12 16:14:00|product_view    |12         |
|2026-08-12 16:52:00|checkout_start  |9          |
|2026-09-03 11:36:00|purchase_success|4          |
|2026-08-12 17:04:00|inventory_update|1          |
|2026-08-12 15:17:00|inventory_update|1          |
|2026-08-12 15:12:00|purchase_success|8          |
|2026-08-12 15:53:00|inventory_update|1          |
|2026-09-03 11:49:00|product_view    |12         |
|2026-08-12 15:44:00|product_vi

In [13]:
def write_country_metrics(batch_df, batch_id):
    target_table = "3_gold.gold_streaming_country_metrics"

    if not spark.catalog.tableExists(target_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("target")
            .merge(
                batch_df.alias("source"),
                "target.country = source.country"
            )
            .whenMatchedUpdate(set={
                "event_count": "source.event_count"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 15, Finished, Available, Finished, False)

In [14]:
country_metrics_query = (
    country_metrics_stream.writeStream
    .outputMode("update")
    .foreachBatch(write_country_metrics)
    .option(
        "checkpointLocation",
        "Files/checkpoints/gold_country_metrics"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 16, Finished, Available, Finished, False)

In [15]:
# Check active streaming queries
for query in spark.streams.active:
    print("Name:", query.name)
    print("Status:", query.status)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 17, Finished, Available, Finished, False)

Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [16]:
# Validate the Streaming Gold country metrics table
spark.read.table("3_gold.gold_streaming_country_metrics").show(
    20,
    truncate=False
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 18, Finished, Available, Finished, False)

+---------------+-----------+
|country        |event_count|
+---------------+-----------+
|Channel Islands|19         |
|United Kingdom |5638       |
|Czech Republic |1          |
|Switzerland    |18         |
|West Indies    |2          |
|Singapore      |5          |
|Israel         |3          |
|Iceland        |2          |
|Portugal       |10         |
|Germany        |114        |
|Belgium        |16         |
|Austria        |4          |
|Finland        |6          |
|Cyprus         |15         |
|France         |95         |
|Canada         |2          |
|Spain          |33         |
|Italy          |14         |
|Japan          |5          |
|RSA            |1          |
+---------------+-----------+
only showing top 20 rows



### 6.1 Real-Time KPI Delta Sink

Write the transformed real-time KPI stream to the `3_gold.gold_streaming_realtime_kpis` Delta table.

The sink maintains one record for each one-minute event-time window and updates the KPI values as additional events arrive within the allowed watermark period.

A dedicated checkpoint location is used to maintain the streaming state and support reliable recovery.

In [17]:
def write_realtime_kpis(batch_df, batch_id):
    target_table = "3_gold.gold_streaming_realtime_kpis"

    if not spark.catalog.tableExists(target_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("target")
            .merge(
                batch_df.alias("source"),
                "target.event_minute = source.event_minute"
            )
            .whenMatchedUpdate(set={
                "total_events": "source.total_events",
                "product_views": "source.product_views",
                "cart_actions": "source.cart_actions",
                "checkout_starts": "source.checkout_starts",
                "purchases": "source.purchases",
                "checkout_to_purchase_rate": "source.checkout_to_purchase_rate",
                "avg_processing_delay_seconds": "source.avg_processing_delay_seconds"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 19, Finished, Available, Finished, False)

In [18]:
realtime_kpis_query = (
    realtime_kpis_stream.writeStream
    .outputMode("update")
    .foreachBatch(write_realtime_kpis)
    .option(
        "checkpointLocation",
        "Files/checkpoints/gold_realtime_kpis"
    )
    .trigger(processingTime="10 seconds")
    .start()
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 20, Finished, Available, Finished, False)

In [19]:
# Check active Streaming Gold queries
for query in spark.streams.active:
    print("Name:", query.name)
    print("Status:", query.status)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 21, Finished, Available, Finished, False)

Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Name: None
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [20]:
spark.read.table("3_gold.gold_streaming_realtime_kpis").show(
    20,
    truncate=False
)

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 22, Finished, Available, Finished, False)

+-------------------+------------+-------------+------------+---------------+---------+-------------------------+----------------------------+
|event_minute       |total_events|product_views|cart_actions|checkout_starts|purchases|checkout_to_purchase_rate|avg_processing_delay_seconds|
+-------------------+------------+-------------+------------+---------------+---------+-------------------------+----------------------------+
|2026-08-12 16:06:00|30          |10           |7           |4              |6        |150.0                    |4.966666666666667           |
|2026-08-12 16:37:00|29          |11           |5           |5              |5        |100.0                    |4.413793103448276           |
|2026-08-12 16:14:00|30          |12           |7           |5              |5        |100.0                    |4.866666666666666           |
|2026-08-12 17:12:00|30          |7            |15          |3              |4        |133.33333333333331       |4.266666666666667           |

### 7. Validate Streaming Gold

Validate all Streaming Gold Delta tables after the streaming queries have processed incoming Silver events.

The validation confirms that:

- All three Gold tables exist in the `3_gold` schema.
- The tables contain streaming data.
- The expected analytical columns are present.
- Event metrics are aggregated by event type and minute.
- Country metrics contain event activity by country.
- Real-time KPI metrics contain the expected event counts, conversion rate, and processing delay.

This validation confirms that the Streaming Gold layer is successfully receiving and transforming data from the Silver streaming layer.

In [21]:
# List the Streaming Gold tables
gold_tables = [
    "3_gold.gold_streaming_event_metrics",
    "3_gold.gold_streaming_country_metrics",
    "3_gold.gold_streaming_realtime_kpis"
]

for table in gold_tables:
    print(f"{table}: {spark.catalog.tableExists(table)}")
    

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 23, Finished, Available, Finished, False)

3_gold.gold_streaming_event_metrics: True
3_gold.gold_streaming_country_metrics: True
3_gold.gold_streaming_realtime_kpis: True


In [22]:
# Check row counts for all Streaming Gold tables
for table in gold_tables:
    row_count = spark.read.table(table).count()
    print(f"{table}: {row_count:,} rows")

StatementMeta(, 39cf4d72-383f-45bc-b74e-17ace54dcb31, 24, Finished, Available, Finished, False)

3_gold.gold_streaming_event_metrics: 1,005 rows
3_gold.gold_streaming_country_metrics: 33 rows
3_gold.gold_streaming_realtime_kpis: 212 rows


### 8. Streaming Gold Layer Complete

The Streaming Gold layer has been successfully implemented using PySpark Structured Streaming and Delta tables.

The layer contains three analytical outputs:

- `gold_streaming_event_metrics` – event activity aggregated by event type and one-minute event-time windows.
- `gold_streaming_country_metrics` – cumulative streaming event activity by country.
- `gold_streaming_realtime_kpis` – one-minute real-time KPIs including event volume, event-type activity, checkout-to-purchase rate, and average processing delay.

All three Gold Delta tables are stored under the `3_gold` schema in the Lakehouse and have been validated as populated streaming outputs.

The Streaming Gold layer is now ready for downstream analytical consumption in Power BI.